# Experiment 2: YOLO + PrismGhost for 20-Class Plant Disease Object Detection

This notebook provides the complete, self-contained, error-free implementation of Experiment 2:
- **Hardware Acceleration**: Local GPU (NVIDIA RTX series) with PyTorch CUDA
- **Datasets**: PlantVillage and PlantDoc combined and mapped to 20 unified classes
- **Custom Architecture**: PrismGhost feature generation module registered with Ultralytics YOLOv8
- **Reproducibility**: Global random seed 42 across all components
- **Training & Validation**: 25 epochs with AdamW, Cosine LR, and AMP
- **Evaluation & Export**: Test set evaluation, bounding box prediction visualization, and ONNX export

## 1. Compute Environment & Hardware Diagnostics

In [ ]:
import os
import sys
import math
import time
import random
import shutil
from pathlib import Path
from collections import Counter

import yaml
import torch
import torch.nn as nn
import numpy as np

print("=" * 70)
print("HARDWARE AND COMPUTE DIAGNOSTICS")
print("=" * 70)
print(f"Python Version  : {sys.version}")
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
print(f"CUDA Version    : {torch.version.cuda}")

if torch.cuda.is_available():
    print(f"GPU Device Name : {torch.cuda.get_device_name(0)}")
    DEVICE = 0
else:
    print("Using CPU")
    DEVICE = "cpu"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

WORKSPACE_ROOT = Path.cwd()
DATASET_ROOT = WORKSPACE_ROOT / "dataset"
EXPERIMENTS_ROOT = WORKSPACE_ROOT / "AgriYOLO_Experiments"
DATASET_ROOT.mkdir(parents=True, exist_ok=True)
EXPERIMENTS_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Workspace Root  : {WORKSPACE_ROOT}")
print(f"Dataset Root    : {DATASET_ROOT}")
print(f"Experiments Root: {EXPERIMENTS_ROOT}")

## 2. Kaggle Dataset Setup & Extraction

In [ ]:
PV_RAW_DIR = DATASET_ROOT / "raw_plantvillage"
PD_RAW_DIR = DATASET_ROOT / "raw_plantdoc"
PV_RAW_DIR.mkdir(parents=True, exist_ok=True)
PD_RAW_DIR.mkdir(parents=True, exist_ok=True)

kaggle_token_file = Path.home() / ".kaggle" / "access_token"
if kaggle_token_file.exists() and "KAGGLE_API_TOKEN" not in os.environ:
    os.environ["KAGGLE_API_TOKEN"] = kaggle_token_file.read_text().strip()

import kaggle
pv_dataset_folder = PV_RAW_DIR / "PlantVillage_for_object_detection" / "Dataset"
if not (pv_dataset_folder / "images").exists():
    print("Downloading PlantVillage dataset...")
    kaggle.api.dataset_download_files("sebastianpalaciob/plantvillage-for-object-detection-yolo", path=str(PV_RAW_DIR), unzip=True)
else:
    print("PlantVillage dataset already downloaded.")

if not (PD_RAW_DIR / "images").exists():
    print("Downloading PlantDoc dataset...")
    kaggle.api.dataset_download_files("yusufmurtaza01/plantdoc-object-detection-dataset", path=str(PD_RAW_DIR), unzip=True)
else:
    print("PlantDoc dataset already downloaded.")

## 3. Unified 20-Class Discovery & Label Remapping

In [ ]:
TARGET_CLASSES = [
    "Apple Scab",
    "Apple Cedar Rust",
    "Apple Healthy",
    "Corn Gray Leaf Spot",
    "Corn Common Rust",
    "Corn Northern Leaf Blight",
    "Grape Black Rot",
    "Grape Healthy",
    "Pepper Bacterial Spot",
    "Pepper Healthy",
    "Potato Early Blight",
    "Potato Late Blight",
    "Tomato Bacterial Spot",
    "Tomato Early Blight",
    "Tomato Late Blight",
    "Tomato Leaf Mold",
    "Tomato Septoria Leaf Spot",
    "Tomato Yellow Leaf Curl Virus",
    "Tomato Mosaic Virus",
    "Tomato Healthy"
]

# Parse PlantVillage
with open(pv_dataset_folder / "classes.yaml", "r", encoding="utf-8") as f:
    pv_yaml_data = yaml.safe_load(f)
pv_names = pv_yaml_data.get("names", [])
pv_class_map = {}
for src_id, name in enumerate(pv_names):
    n = name.lower()
    if "apple___apple_scab" in n: pv_class_map[src_id] = 0
    elif "apple___cedar_apple_rust" in n: pv_class_map[src_id] = 1
    elif "apple___healthy" in n: pv_class_map[src_id] = 2
    elif "cercospora_leaf_spot" in n or "gray_leaf_spot" in n: pv_class_map[src_id] = 3
    elif "corn___common_rust" in n: pv_class_map[src_id] = 4
    elif "corn___northern_leaf_blight" in n: pv_class_map[src_id] = 5
    elif "grape___black_rot" in n: pv_class_map[src_id] = 6
    elif "grape___healthy" in n: pv_class_map[src_id] = 7
    elif "pepper" in n and "bacterial_spot" in n: pv_class_map[src_id] = 8
    elif "pepper" in n and "healthy" in n: pv_class_map[src_id] = 9
    elif "potato___early_blight" in n: pv_class_map[src_id] = 10
    elif "potato___late_blight" in n: pv_class_map[src_id] = 11
    elif "tomato___bacterial_spot" in n: pv_class_map[src_id] = 12
    elif "tomato___early_blight" in n: pv_class_map[src_id] = 13
    elif "tomato___late_blight" in n: pv_class_map[src_id] = 14
    elif "tomato___leaf_mold" in n: pv_class_map[src_id] = 15
    elif "tomato___septoria_leaf_spot" in n: pv_class_map[src_id] = 16
    elif "tomato_yellow_leaf_curl_virus" in n: pv_class_map[src_id] = 17
    elif "tomato_mosaic_virus" in n: pv_class_map[src_id] = 18
    elif "tomato___healthy" in n: pv_class_map[src_id] = 19

# Parse PlantDoc
with open(PD_RAW_DIR / "dataset.yaml", "r", encoding="utf-8") as f:
    pd_yaml_data = yaml.safe_load(f)
pd_names = pd_yaml_data.get("names", {})
pd_class_map = {}
for src_id, name in (pd_names.items() if isinstance(pd_names, dict) else enumerate(pd_names)):
    src_id = int(src_id)
    n = name.lower()
    if "apple scab" in n: pd_class_map[src_id] = 0
    elif "apple rust" in n: pd_class_map[src_id] = 1
    elif "apple leaf" in n: pd_class_map[src_id] = 2
    elif "corn gray leaf spot" in n: pd_class_map[src_id] = 3
    elif "corn rust leaf" in n: pd_class_map[src_id] = 4
    elif "corn leaf blight" in n: pd_class_map[src_id] = 5
    elif "grape leaf black rot" in n: pd_class_map[src_id] = 6
    elif n == "grape leaf": pd_class_map[src_id] = 7
    elif "bell_pepper leaf spot" in n: pd_class_map[src_id] = 8
    elif n == "bell_pepper leaf": pd_class_map[src_id] = 9
    elif "potato leaf early blight" in n: pd_class_map[src_id] = 10
    elif "potato leaf late blight" in n: pd_class_map[src_id] = 11
    elif "tomato leaf bacterial spot" in n: pd_class_map[src_id] = 12
    elif "tomato early blight leaf" in n: pd_class_map[src_id] = 13
    elif "tomato leaf late blight" in n: pd_class_map[src_id] = 14
    elif "tomato mold leaf" in n: pd_class_map[src_id] = 15
    elif "tomato septoria leaf spot" in n: pd_class_map[src_id] = 16
    elif "tomato leaf yellow virus" in n: pd_class_map[src_id] = 17
    elif "tomato leaf mosaic virus" in n: pd_class_map[src_id] = 18
    elif n == "tomato leaf": pd_class_map[src_id] = 19

print(f"Mapped {len(pv_class_map)} classes from PlantVillage and {len(pd_class_map)} from PlantDoc.")

## 4. Collision-Safe Merge, Validation & 80/10/10 Split

In [ ]:
def parse_yolo_labels(lbl_path, class_map):
    valid_boxes = []
    if not lbl_path.exists():
        return valid_boxes
    with open(lbl_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            try:
                src_cls = int(float(parts[0]))
                xc, yc, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            except (ValueError, TypeError):
                continue
            if src_cls not in class_map:
                continue
            if not (0.0 <= xc <= 1.0 and 0.0 <= yc <= 1.0 and 0.0 < w <= 1.0 and 0.0 < h <= 1.0):
                continue
            valid_boxes.append((class_map[src_cls], xc, yc, w, h))
    return valid_boxes

merged_records = []
pv_img_dir = pv_dataset_folder / "images"
pv_lbl_dir = pv_dataset_folder / "labels"
for img_path in pv_img_dir.glob("*.*"):
    if img_path.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]:
        lbl_path = pv_lbl_dir / (img_path.stem + ".txt")
        boxes = parse_yolo_labels(lbl_path, pv_class_map)
        if boxes:
            merged_records.append({"orig_img": img_path, "new_stem": f"pv_dataset_{img_path.stem}", "suffix": img_path.suffix.lower(), "boxes": boxes})

for split in ["train", "val"]:
    pd_img_dir = PD_RAW_DIR / "images" / split
    pd_lbl_dir = PD_RAW_DIR / "labels" / split
    for img_path in pd_img_dir.glob("*.*"):
        if img_path.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]:
            lbl_path = pd_lbl_dir / (img_path.stem + ".txt")
            boxes = parse_yolo_labels(lbl_path, pd_class_map)
            if boxes:
                merged_records.append({"orig_img": img_path, "new_stem": f"pd_{split}_{img_path.stem}", "suffix": img_path.suffix.lower(), "boxes": boxes})

print(f"Total valid mapped samples: {len(merged_records)}")
random.seed(SEED)
random.shuffle(merged_records)

n_total = len(merged_records)
n_train = int(0.80 * n_total)
n_val = int(0.10 * n_total)
splits = {
    "train": merged_records[:n_train],
    "val": merged_records[n_train:n_train + n_val],
    "test": merged_records[n_train + n_val:]
}

for split_name, records in splits.items():
    d_img = DATASET_ROOT / "images" / split_name
    d_lbl = DATASET_ROOT / "labels" / split_name
    d_img.mkdir(parents=True, exist_ok=True)
    d_lbl.mkdir(parents=True, exist_ok=True)
    for rec in records:
        dest_img = d_img / f"{rec['new_stem']}{rec['suffix']}"
        dest_lbl = d_lbl / f"{rec['new_stem']}.txt"
        if not dest_img.exists():
            shutil.copy2(rec["orig_img"], dest_img)
        with open(dest_lbl, "w", encoding="utf-8") as f:
            for b in rec["boxes"]:
                f.write(f"{b[0]} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}\n")

data_yaml_path = DATASET_ROOT / "data.yaml"
with open(data_yaml_path, "w", encoding="utf-8") as f:
    yaml.dump({
        "path": str(DATASET_ROOT.resolve()).replace("\\", "/"),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "nc": 20,
        "names": {i: name for i, name in enumerate(TARGET_CLASSES)}
    }, f, sort_keys=False)
print(f"data.yaml generated: {data_yaml_path}")

## 5. PrismGhost Module Registration & Architecture YAML

In [ ]:
class PrismGhost(nn.Module):
    def __init__(self, c1, c2, k=3, s=1, ratio=2):
        super().__init__()
        primary = math.ceil(c2 / ratio)
        cheap = primary * (ratio - 1)
        self.primary = nn.Sequential(
            nn.Conv2d(c1, primary, k, s, k // 2, bias=False),
            nn.BatchNorm2d(primary),
            nn.SiLU(),
        )
        self.cheap = nn.Sequential(
            nn.Conv2d(primary, cheap, 3, 1, 1, groups=primary, bias=False),
            nn.BatchNorm2d(cheap),
            nn.SiLU(),
        )
        self.c2 = c2

    def forward(self, x):
        y = self.primary(x)
        return torch.cat((y, self.cheap(y)), 1)[:, :self.c2]

import ultralytics.nn.modules as ultralytics_modules
import ultralytics.nn.tasks as ultralytics_tasks
ultralytics_modules.PrismGhost = PrismGhost
ultralytics_tasks.PrismGhost = PrismGhost
setattr(ultralytics_modules, "PrismGhost", PrismGhost)
setattr(ultralytics_tasks, "PrismGhost", PrismGhost)

model_yaml_content = """nc: 20
scales:
  n: [1.0, 1.0, 1024]

backbone:
  - [-1, 1, PrismGhost, [16, 3, 2]] # 0-P1/2 (downsample 640x640 -> 320x320)
  - [-1, 1, PrismGhost, [32, 3, 2]] # 1-P2/4 (downsample 320x320 -> 160x160)
  - [-1, 1, PrismGhost, [64, 3, 2]] # 2-P3/8 (downsample 160x160 -> 80x80)
  - [-1, 1, PrismGhost, [128, 3, 2]] # 3-P4/16 (downsample 80x80 -> 40x40)
  - [-1, 1, PrismGhost, [256, 3, 2]] # 4-P5/32 (downsample 40x40 -> 20x20)
  - [-1, 1, SPPF, [256, 5]] # 5-P5/32

head:
  # Top-down feature aggregation
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]] # 6
  - [[-1, 3], 1, Concat, [1]] # 7 cat backbone P4
  - [-1, 1, PrismGhost, [128, 3, 1]] # 8 (P4/16)
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]] # 9
  - [[-1, 2], 1, Concat, [1]] # 10 cat backbone P3
  - [-1, 1, PrismGhost, [64, 3, 1]] # 11 (P3/8)
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]] # 12
  - [[-1, 1], 1, Concat, [1]] # 13 cat backbone P2
  - [-1, 1, PrismGhost, [32, 3, 1]] # 14 (P2/4 head output)

  # Bottom-up feature aggregation
  - [-1, 1, PrismGhost, [64, 3, 2]] # 15 downsample P2->P3
  - [[-1, 11], 1, Concat, [1]] # 16 cat P3
  - [-1, 1, PrismGhost, [64, 3, 1]] # 17 (P3/8 head output)
  - [-1, 1, PrismGhost, [128, 3, 2]] # 18 downsample P3->P4
  - [[-1, 8], 1, Concat, [1]] # 19 cat P4
  - [-1, 1, PrismGhost, [128, 3, 1]] # 20 (P4/16 head output)
  - [-1, 1, PrismGhost, [256, 3, 2]] # 21 downsample P4->P5
  - [[-1, 5], 1, Concat, [1]] # 22 cat P5
  - [-1, 1, PrismGhost, [256, 3, 1]] # 23 (P5/32 head output)

  # Quad-head Detect (P2, P3, P4, P5)
  - [[14, 17, 20, 23], 1, Detect, [nc]] # 24 Detect(P2, P3, P4, P5)
"""
model_yaml_path = WORKSPACE_ROOT / "yolo_prismghost.yaml"
with open(model_yaml_path, "w", encoding="utf-8") as f:
    f.write(model_yaml_content)
print(f"Saved {model_yaml_path}")

## 6. Forward Pass Validation (Zero NaN / Inf)

In [ ]:
from ultralytics import YOLO
model = YOLO(str(model_yaml_path))
test_dev = "cuda:0" if torch.cuda.is_available() else "cpu"
model.to(test_dev)

dummy = torch.zeros((1, 3, 640, 640), device=test_dev)
with torch.no_grad():
    out = model.model(dummy)

print("Forward pass executed:")
for idx, o in enumerate(out):
    print(f"  Scale [{idx}] output shape: {o.shape}")
    assert not torch.isnan(o).any(), f"NaN found in scale {idx}"
    assert not torch.isinf(o).any(), f"Inf found in scale {idx}"
print("Assertion passed: No NaN or Inf values produced.")

## 7. Model Training (25 Epochs, AdamW, Cosine LR, AMP)

In [ ]:
results = model.train(
    data=str(data_yaml_path.resolve()).replace("\\", "/"),
    epochs=25,
    imgsz=640,
    batch=16,
    lr0=0.001,
    optimizer="AdamW",
    cos_lr=True,
    amp=torch.cuda.is_available(),
    device=DEVICE,
    workers=4,
    seed=42,
    project=str(EXPERIMENTS_ROOT.resolve()).replace("\\", "/"),
    name="yolo_prismghost_run",
    exist_ok=True
)

## 8. Test Set Evaluation & Metrics

In [ ]:
best_weights = EXPERIMENTS_ROOT / "yolo_prismghost_run" / "weights" / "best.pt"
eval_model = YOLO(str(best_weights))
metrics = eval_model.val(
    data=str(data_yaml_path.resolve()).replace("\\", "/"),
    split="test",
    imgsz=640,
    batch=16,
    device=DEVICE
)
print(f"mAP@50    : {metrics.box.map50:.4f}")
print(f"mAP@50-95 : {metrics.box.map:.4f}")
print(f"Precision : {metrics.box.mp:.4f}")
print(f"Recall    : {metrics.box.mr:.4f}")

## 9. Visual Inference & ONNX Model Export

In [ ]:
# Test inference
test_img_dir = DATASET_ROOT / "images" / "test"
eval_model.predict(
    source=str(test_img_dir.resolve()).replace("\\", "/"),
    save=True,
    imgsz=640,
    device=DEVICE,
    project=str(EXPERIMENTS_ROOT.resolve()).replace("\\", "/"),
    name="test_predictions",
    exist_ok=True
)

# ONNX Export
onnx_file = eval_model.export(format="onnx", imgsz=640, dynamic=True)
print(f"ONNX Model Exported: {onnx_file}")
assert Path(onnx_file).exists() and Path(onnx_file).stat().st_size > 0
print("ONNX Export Verified successfully!")